In [ ]:

import sys
from pathlib import Path
import os
import yaml
import echopype as ep
from aa_si_visualization import echogram

# Add the package source directory to path so aa_si_calibration is importable
# (not needed if the package is installed via pip install -e .)
sys.path.insert(0, str(Path("../src").resolve()))

from aa_si_calibration.calibration import generate_standardized_cal_mapping

%load_ext autoreload
%autoreload 2

from aa_si_calibration import calibration
from aa_si_calibration import comparison

from aa_si_ml import ml
from aa_si_utils import utils
from aa_si_utils.data_retrieval import query_ncei_data, download_ncei_data


print("All imports successful!")



In [ ]:

# =============================================================================
# LOAD RECIPE CONFIGURATION
# =============================================================================

RECIPE_PATH = "./nefsc_us_e2_recipe.yaml"

with open(RECIPE_PATH, "r") as f:
    config = yaml.safe_load(f)

# Extract top-level config sections
env = config["local_environment"]
cal_cfg = config["calibration"]["calibration_pipeline"]
masking_cfg = config["masking"]
noise_cfg = config["noise_removal"]
mvbs_cfg = config["mvbs"]
line_cfg = config["line_files"]
echogram_cfg = config["echogram_visualization"]
sv_comp_cfg = config["sv_comparison"]
ml_cfg = config["ml"]
retrieval_cfg = config["data_retrieval"]

# Sub-sections
setup = env["initial_setup"]
raw_combine_cfg = env["open_and_combine_raw_files"]

# Derived values
global_params = {
    "cruise_id": cal_cfg["cruise_id"],
    "record_author": cal_cfg["record_author"],
}

# Build overlay_lines from config
overlay_lines = [
    {
        "var": f'{line_cfg["line_name"]}{line["var_suffix"]}',
        "style": line["style"],
    }
    for line in line_cfg["overlay_lines"]
]

print(f"Recipe loaded from: {RECIPE_PATH}")
print(f"Record author: {cal_cfg['record_author']}")
print(f"Cruise ID: {cal_cfg['cruise_id']}")
print(f"Short filenames: {cal_cfg['short_filenames']}")
print(f"Keep unused standardized files: {cal_cfg['keep_unused_standardized_files']}")
print(f"Conflict resolution mode: {cal_cfg['conflict_resolution']}")
print(f"\nInput folders:")
print(f"  Raw files:        {setup['raw_input_folder']}")
print(f"  Calibration files: {setup['cal_input_folder']}")
print(f"\nOutput base: {setup['output_base']}")



In [ ]:
# Query NCEI for raw files
results = query_ncei_data(
    file_time_start=retrieval_cfg["file_time_start"],
    file_time_end=retrieval_cfg["file_time_end"],
    filters=retrieval_cfg["filters"],
)

# Preview results
for r in results:
    print(f"  {r['FILE_NAME']}  ({r['FILE_DATETIME']})")


In [ ]:
# Download raw files (plus .bot and .idx companions)
downloaded_paths = download_ncei_data(
    results,
    output_dir=setup["raw_input_folder"],
    companion_extensions=retrieval_cfg["companion_extensions"],
)


In [ ]:

raw_file_paths = utils.initial_setup_and_validation(
    raw_input_folder=setup["raw_input_folder"],
    netcdf_output_folder_string=setup["netcdf_output_folder"],
    sv_output_folder_string=setup["sv_output_folder"],
    output_logs_folder_string=setup["output_logs_folder"],
    raw_file_names=setup["raw_file_names"],
    clear_previous_json_logs=setup["clear_previous_json_logs"],
)



In [ ]:
# =============================================================================
# STEPS 1-3 + VERIFICATION: Full calibration pipeline
# =============================================================================
# 1. Read raw file configurations and save them
# 2. Read manufacturer calibration files and save to single-channel standardized files
# 3. Load single-channel files, match raw channels to calibration data, save mapping
# 4. Verify required parameters and file usage

pipeline_result = generate_standardized_cal_mapping(
    raw_input_folder=setup["raw_input_folder"],
    cal_input_folder=setup["cal_input_folder"],
    output_base=setup["output_base"],
    global_params=global_params,
    short_filenames=cal_cfg["short_filenames"],
    keep_unused=cal_cfg["keep_unused_standardized_files"],
    conflict_resolution=cal_cfg["conflict_resolution"],
)

mapping_dict = pipeline_result["mapping_dict"]
calibration_dict = pipeline_result["calibration_dict"]

In [ ]:

echodata = utils.open_and_combine_raw_files(
    raw_file_paths,
    setup["netcdf_output_folder"],
    sonar_model=raw_combine_cfg["sonar_model"],
    include_bot=raw_combine_cfg["include_bot"],
)



In [ ]:

# extract calibration parameters from original netcdf file
original_params = calibration.extract_netcdf_calibration_parameters(echodata, setup["output_logs_folder"])

# the calibration parameters come from the first file in the sequential raw files that were combined by echopype
nc_cal_files = [os.path.basename(raw_file_paths[0])]

original_params["other_params"]["source_filenames_across_channels"] = nc_cal_files
original_params["other_params"]["source_file_type"] = ".raw"

# print parameters from netcdf file
calibration.print_calibration_values(echodata, original_params, "Calibration Values From .nc File")



In [ ]:

# =============================================================================
# Extract standardized calibration parameters in comparison format
# =============================================================================
# Convert the per-channel standardized data from calibration_dict into the
# (cal_params, env_params, other_params) structure needed by
# comparison.run_full_calibration_comparison.

standardized_params = calibration.extract_standardized_calibration_parameters(
    calibration_dict, mapping_dict, echodata=echodata,
)

standardized_params["other_params"]["source_filenames_across_channels"] = [os.path.basename(raw_file_paths[0])]
standardized_params["other_params"]["source_file_type"] = ".cal" #TODO: derive this from files in calibration folder

calibration.print_calibration_values(echodata, standardized_params, "Calibration Values From Standardized Files")



In [ ]:
comparison.compare_calibration_parameters(standardized_params, original_params, echodata)

In [ ]:


# Compute baseline Sv (no calibration overrides)
ds_Sv_baseline_unmasked = ep.calibrate.compute_Sv(echodata)

# Create mask: seafloor + surface + frequency exclusions
mask_params = masking_cfg["create_data_mask"]
mask = utils.create_data_mask(
    echodata,
    ds_Sv_baseline_unmasked,
    seafloor_buffer_m=mask_params["seafloor_buffer_m"],
    frequencies_to_mask=mask_params["frequencies_to_mask"],
)

In [ ]:
# Compute Sv with each individual calibration parameter and combined
sv_datasets = comparison.compute_calibrated_sv_datasets(echodata, standardized_params)

In [ ]:
# Apply mask to baseline and all calibrated Sv datasets
ds_Sv_baseline = utils.apply_mask_to_sv(ds_Sv_baseline_unmasked, mask)
calibrated_sv = utils.apply_mask_to_sv_datasets(sv_datasets, mask)

ds_Sv_baseline.to_netcdf(setup["sv_output_folder"] + "/NEFSC_processed_data.nc")
print("Mask applied and baseline saved")

In [ ]:
# Run Sv comparison analysis: effects calculation, plots, range analysis, echograms, additive verification
comparison_results = comparison.run_sv_comparison_analysis(
    ds_Sv_baseline,
    calibrated_sv,
    echodata,
    original_params,
    setup["output_logs_folder"],
    sv_flag_thresholds=sv_comp_cfg["sv_flag_thresholds"],
)

ds_Sv_calibrated = calibrated_sv["combined"]
verification_results = comparison_results["verification_results"]

In [ ]:

noise_params = noise_cfg["remove_noise"]
ds_Sv_clean = ml.remove_noise(
    ds_Sv_baseline,
    noise_range_sample_num=noise_params["noise_range_sample_num"],
    noise_ping_num=noise_params["noise_ping_num"],
)

mvbs_params = mvbs_cfg["compute_mvbs"]
if mvbs_params.get("mvbs_nan_threshold") is not None:
    ds_Sv_clean = utils.mask_sparse_bins(
        ds_Sv_clean,
        range_bin=mvbs_params["mvbs_range_bin"],
        ping_time_bin=mvbs_params["mvbs_ping_time_bin"],
        nan_threshold=mvbs_params["mvbs_nan_threshold"],
    )

ds_MVBS = ep.commongrid.compute_MVBS(
    ds_Sv_clean,
    range_bin=mvbs_params["mvbs_range_bin"],
    ping_time_bin=mvbs_params["mvbs_ping_time_bin"],
)

ds_MVBS = utils.add_dive_profile_to_dataset(
    ds_MVBS,
    line_cfg["line_file_path"],
    line_cfg["line_name"],
)

echo_params = echogram_cfg["plot_sv_echogram"]

echogram.plot_sv_echogram(
    ds_Sv_clean,
    min_depth=echo_params["min_depth"],
    max_depth=echo_params["max_depth"],
    ping_min=echo_params["ping_min"],
    ping_max=echo_params["ping_max"],
    x_axis_units=echo_params["x_axis_units"],
    y_axis_units=echo_params["y_axis_units"],
    echodata=echodata,
)

echogram.plot_sv_echogram(
    ds_MVBS,
    ds_Sv_clean,
    min_depth=echo_params["min_depth"],
    max_depth=echo_params["max_depth"],
    ping_min=echo_params["ping_min"],
    ping_max=echo_params["ping_max"],
    x_axis_units=echo_params["x_axis_units"],
    y_axis_units=echo_params["y_axis_units"],
    echodata=echodata,
    overlay_lines=overlay_lines,
)


In [ ]:
reshape_cfg = ml_cfg["reshape_data_for_ml"]
norm_cfg = ml_cfg["normalize_data"]
hdbscan_cfg = ml_cfg["hdbscan"]

# Step 1: Reshape data for ML
ds_ml_ready = ml.reshape_data_for_ml(
    ds_MVBS,
    data_var=reshape_cfg["data_var"],
    dataset_name=reshape_cfg["dataset_name"],
    feature_strategy=reshape_cfg["feature_strategy"],
    baseline_channel=reshape_cfg["baseline_channel"],
)

# Step 2: Normalize data
ds_normalized = ml.normalize_data(
    ds_ml_ready,
    method=norm_cfg["method"],
    shift_positive=norm_cfg["shift_positive"],
    dataset_name=reshape_cfg["dataset_name"],
    normalization_name=norm_cfg["normalization_name"],
    feature_weights=norm_cfg["feature_weights"],
)

# Step 3: Plot normalized data echogram
echogram.plot_flattened_data_echogram(
    ds_normalized,
    reshape_cfg["dataset_name"],
    ds_Sv_clean,
    ml_specific_data_name=norm_cfg["normalization_name"],
    min_depth=echo_params["min_depth"],
    max_depth=echo_params["max_depth"],
    ping_min=echo_params["ping_min"],
    ping_max=echo_params["ping_max"],
    x_axis_units=echo_params["x_axis_units"],
    y_axis_units=echo_params["y_axis_units"],
)

# Step 4: Run HDBSCAN clustering
ds_MVBS, gridded_background_results_dbscan, dbscan_results = ml.extract_data_and_run_hdbscan(
    ds_normalized,
    reshape_cfg["dataset_name"],
    ds_Sv_original=ds_Sv_clean,
    custom_normalization_name=norm_cfg["normalization_name"],
    ml_result_name=hdbscan_cfg["ml_result_name"],
    plot_window=hdbscan_cfg["plot_window"],
    min_samples=hdbscan_cfg["min_samples"],
    sample_size=hdbscan_cfg["sample_size"],
    min_cluster_size=hdbscan_cfg["min_cluster_size"],
    cluster_selection_method=hdbscan_cfg["cluster_selection_method"],
    use_hdbscan=hdbscan_cfg["use_hdbscan"],
    cluster_colors=ml_cfg["cluster_colors"],
    overlay_line_var=line_cfg["line_name"],
)